# Lab B — Sequential Monte Carlo and Sampling Explorer

**September 29, 2026 · Advanced Machine Learning · CPU runtime.**

Save a copy in Drive, then run from top to bottom. Read each prediction before running its experiment.
Setup checks package versions and verifies downloaded source hashes. If it asks for a restart, restart
the session once and run all cells again. No manual dataset upload, account key, or GPU is needed.
All data are transparently generated synthetic examples.

Exercises are independent: an unfinished response must not prevent later demonstrations.
Use the source links before calling an unfamiliar API. A permanent notebook link is different from
a temporary Gradio share URL, which requires an active runtime.


We track one synthetic latent trajectory and compare SIS and bootstrap particle filtering with an exact Kalman reference. Filtering uses observations up to the current time; this is not smoothing. [Main lesson](https://github.com/lunalab-ai/advML/blob/2026-fall-w5/course/notion/w5/w5-monte-carlo-inference.md) · [Experiment worksheet](https://github.com/lunalab-ai/advML/blob/2026-fall-w5/course/notion/w5/w5-experiment-record.md)


In [ ]:
import sys, subprocess, importlib.metadata, hashlib, urllib.request, time
from pathlib import Path

PINS = {"numpy":"2.2.6", "matplotlib":"3.10.3", "scipy":"1.16.3",
        "pandas":"2.3.3", "arviz":"0.22.0", "gradio":"6.27.0"}
missing = []
for package, version in PINS.items():
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        missing.append(f"{package}=={version}")
restart = [p for p,v in PINS.items() if p in sys.modules and getattr(sys.modules[p], "__version__", v) != v]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
if restart:
    raise RuntimeError("Packages updated. Use Runtime > Restart session, then Run all. Loaded old packages: "+", ".join(restart))

CODE_REF = "2026-fall-w5"
EXPECTED = {'advml_vi.py': '699c3e5f7f75bee9dfdceff05e4956f7bd2bf0b9abff6c0eacf11a9ac3f1ad2a', 'advml_sampling.py': '7b881dd17a5895eb6c44835ae6e6516e2448258f99a76965a6b9d352049b0f1e', 'advml_smc.py': '890d797f1adaf009a5d46bf36058c5b8306654abfc6228412426486dc302deaa', 'advml_sampling_app.py': 'd4bfb72bb53c3c18908eb87e513bbf7a9cd5de050c0e4db8332948409b8a62ad'}
local = Path.cwd() / "src"
local_ok = all((local/name).is_file() and hashlib.sha256((local/name).read_bytes()).hexdigest()==digest
               for name,digest in EXPECTED.items())
support = local if local_ok else Path.cwd() / "w5_support" / CODE_REF
support.mkdir(parents=True, exist_ok=True)
downloaded = []
for name,digest in EXPECTED.items():
    path = support/name
    if path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest()==digest:
        continue
    url = f"https://raw.githubusercontent.com/lunalab-ai/advML/{CODE_REF}/src/{name}"
    for attempt in range(3):
        try:
            with urllib.request.urlopen(url, timeout=40) as response:
                payload = response.read()
            break
        except (OSError, TimeoutError):
            if attempt == 2:
                raise
            time.sleep(attempt+1)
    if hashlib.sha256(payload).hexdigest()!=digest:
        raise RuntimeError(f"Source hash mismatch: {name}. Do not run unverified code.")
    temporary = path.with_suffix(".download")
    temporary.write_bytes(payload)
    temporary.replace(path)
    downloaded.append(name)
sys.path.insert(0, str(support.resolve()))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr
from IPython.display import display
print({"Python":sys.version.split()[0], "code_ref":CODE_REF, "downloaded":downloaded,
       "cache_or_reviewed_local":not downloaded, "versions":PINS})


## 1. Data and exact reference

### [simulate_tracking](https://github.com/lunalab-ai/advML/blob/2026-fall-w5/src/advml_smc.py#L19)

Generate fixed synthetic truth and observations, both shape (steps,).

steps>=1, finite a, positive noise SDs. Initial state has variance one;
no transition is applied before observation y[0]. Local RNG, no file/network
access. Keep this seed fixed while varying particle-filter seeds.

### [kalman_filter](https://github.com/lunalab-ai/advML/blob/2026-fall-w5/src/advml_smc.py#L39)

Exact scalar Gaussian filtering reference for the stated initial model.

observations: finite 1D y of length T>=1. Return mean/variance/predicted_mean/
predicted_variance/gain arrays (T,), and log_evidence scalar. Variance is
posterior uncertainty, not the Monte Carlo variance of an estimator.
Uses data up through t only (filtering); no future-data smoothing.


In [ ]:
from advml_smc import simulate_tracking, kalman_filter, systematic_resample, particle_filter
from advml_sampling import normalize_log_weights, weight_ess
from advml_sampling_app import filter_view, build_sampling_app
data=simulate_tracking(seed=20260929)
observed=data["observations"]
exact=kalman_filter(observed)
display(pd.DataFrame({"observed":observed,"latent_truth":data["truth"],
                      "exact_filter_mean":exact["mean"],"exact_filter_variance":exact["variance"]}).head())


## B1 — your turn

Before applying the first transition, let the prior be N(0,1), the first observation be 2, and observation SD be 1. Calculate the posterior mean and variance. Explain why multiplying the initial variance by a² first would implement a different model.


In [ ]:
# Write your experiment or calculation here. Later demonstrations do not depend on this cell.


## 2. Bootstrap weights and SIS

### [particle_filter](https://github.com/lunalab-ai/advML/blob/2026-fall-w5/src/advml_smc.py#L86)

Bootstrap SIS/PF with optional resampling AFTER each measurement update.

y: finite 1D length T. particles=N>=2. threshold in [0,1]; zero means SIS.
Resample when pre-resampling weight ESS < threshold*N. Propagation uses the
state transition, so incremental weights are observation likelihoods.
Initial particles are N(0,1), weighted by y[0]. All means/variances and
particle/weight arrays are recorded BEFORE resampling. Return arrays mean,
variance, ess_before, ess_after, unique_roots_after, resampled of shape (T,);
samples, weights, root_labels_before, resampling_indices of shape (T,N).
Roots refer to initial particle IDs; distinct positions do not imply distinct
ancestry. log_evidence estimates log likelihood and is not unbiased in log.
A collapsed or invalid computation raises; no uniform-weight fallback.


In [ ]:
sis=particle_filter(observed,particles=400,threshold=0,seed=113)
pf=particle_filter(observed,particles=400,threshold=.5,seed=113)
def filtering_metrics(run):
    return {"RMSE_vs_Kalman_mean":float(np.sqrt(np.mean((run["mean"]-exact["mean"])**2))),
            "RMSE_vs_truth":float(np.sqrt(np.mean((run["mean"]-data["truth"])**2))),
            "minimum_weight_ESS":float(run["ess_before"].min()),
            "final_ancestors":int(run["unique_roots_after"][-1]),
            "resampling_events":int(run["resampled"].sum())}
display(pd.DataFrame([filtering_metrics(sis),filtering_metrics(pf)],index=["SIS","bootstrap PF"]))


## B2 — your turn

Previous normalized weights are (0.8,0.2); the new bootstrap likelihood values are (0.1,0.9). A buggy SIS implementation normalizes only the likelihood. Calculate the correct new weights and explain when likelihood-only weighting would be valid.


In [ ]:
# Write your experiment or calculation here. Later demonstrations do not depend on this cell.


## 3. Resampling and ancestry

### [systematic_resample](https://github.com/lunalab-ai/advML/blob/2026-fall-w5/src/advml_smc.py#L69)

Return N ancestor indices by systematic resampling of N weights.

weights: finite nonnegative 1D, positive sum; rng is a local NumPy Generator.
Draw one uniform offset in [0,1/N) and use N equally spaced positions.
Output shape (N,), integer indices in [0,N). Consumes rng but does not
mutate weights. Resampled particles are dependent copies, not new evidence.

### [weight_ess](https://github.com/lunalab-ai/advML/blob/2026-fall-w5/src/advml_sampling.py#L112)

Return 1/sum(normalized_weights**2), a WEIGHT-concentration diagnostic.

Accept nonnegative finite 1D weights with positive sum; normalize internally.
This is not MCMC autocorrelation ESS and cannot detect unsampled regions.
Uniform N weights yield N; a single nonzero weight yields 1.


## B3 — your turn

For weights (0.04,0.03,0.06,0.60,0.10,0.07,0.05,0.05), predict what systematic resampling does to weight ESS and distinct ancestors. Run it and explain why copies do not create independent evidence.


In [ ]:
# Write your experiment or calculation here. Later demonstrations do not depend on this cell.


In [ ]:
figure,diagnostics,csv_path=filter_view(400,.5,.5,113)
display(figure);display(diagnostics);plt.close(figure)
print("Timewise CSV:",csv_path)


The plot records filtering means and variances before resampling. A jump in after-resampling ESS comes from uniform weights. The ancestor count traces original particle identities, not just the number of distinct floating-point positions.


## B4 — your turn

At fixed observations, compare N=100, 400, and 1600 over multiple particle seeds. Use RMSE against the exact Kalman mean. State a conclusion about the distribution of errors rather than claiming that every seed improves.


In [ ]:
# Write your experiment or calculation here. Later demonstrations do not depend on this cell.


In [ ]:
records=[]
for n in (100,400,1600):
    for threshold in (0.,.5):
        for seed in range(12):
            result=particle_filter(observed,n,threshold=threshold,seed=seed)
            records.append({"N":n,"threshold":threshold,"particle_seed":seed,**filtering_metrics(result)})
results=pd.DataFrame(records)
display(results.groupby(["N","threshold"])["RMSE_vs_Kalman_mean"].agg(["mean","std","min","max"]))
results.to_csv("w5-filter-comparisons.csv",index=False)


## B5 — your turn

A filter has small error versus the Kalman mean but substantial error versus latent truth. Is that a numerical failure? Explain the shaded Kalman interval and why a large after-resampling ESS does not settle the question.


In [ ]:
# Write your experiment or calculation here. Later demonstrations do not depend on this cell.


## 4. Connect controls to a callback

### [filter_view](https://github.com/lunalab-ai/advML/blob/2026-fall-w5/src/advml_sampling_app.py#L157)

Gradio callback -> Figure, metric dict, timewise CSV path for tracking.

particles>=2, observation_sd>0, resampling threshold in [0,1], integer seed
for PARTICLES. Data seed stays 20260929. Changing observation_sd rescales
the same simulated observation noise and changes the assumed likelihood;
changing particles/threshold/particle seed leaves data identical.
CSV includes truth, observations, Kalman/PF estimates, ESS and ancestor count.
Shows filtering moments before resampling; no future observations are used.

This working small interface holds the threshold at zero. The particle-count slider becomes the callback argument. The returned Figure, metric dictionary, and CSV path go to the three outputs in order.


In [ ]:
def sis_callback(n):
    return filter_view(particles=int(n),observation_sd=.5,threshold=0.,seed=7)
with gr.Blocks(analytics_enabled=False) as sis_app:
    gr.Markdown("# A minimal SIS interface")
    n_control=gr.Slider(50,1000,value=300,step=50,label="Particles")
    button=gr.Button("Run")
    plot=gr.Plot();stats=gr.JSON();download=gr.File()
    button.click(sis_callback,[n_control],[plot,stats,download])


## B6 — your turn

Extend the small interface with a resampling-threshold slider. Write a two-argument callback, connect both inputs in order, and confirm that threshold 0 and threshold 0.8 use identical observations but different resampling behavior. Then use the complete Sampling Explorer below.


In [ ]:
# Write your experiment or calculation here. Later demonstrations do not depend on this cell.


## 5. Complete Sampling Explorer

### [build_sampling_app](https://github.com/lunalab-ai/advML/blob/2026-fall-w5/src/advml_sampling_app.py#L208)

Build (do not launch) a two-tab Gradio Blocks Sampling Explorer.

Return a fresh Blocks object. Buttons map visible controls in the same
order as static_view/filter_view arguments to plot/JSON/CSV outputs.
CPU-only, no account keys, no user uploads. Launch in Colab with share=True;
that temporary URL depends on the active runtime. Keep the notebook link.

Use the fixed-target tab to revisit Lab A, then the tracking tab to compare SIS and PF. Predict, change one setting, inspect, export, and write the evidence paragraph in the worksheet. For Gaussian Gibbs select the Gaussian target; the mixture deliberately has no Gibbs conditional implementation in this lab.


In [ ]:
app=build_sampling_app()
assert len(app.fns)==2
np.testing.assert_allclose(pf["weights"].sum(1),1.,atol=1e-12)
assert np.all(np.diff(pf["unique_roots_after"])<=0)
assert not sis["resampled"].any()
np.testing.assert_allclose(pf["ess_after"][pf["resampled"]],400)
fig,metrics,path=filter_view(150,.5,.5,10)
exported=pd.read_csv(path)
assert len(exported)==40 and np.isfinite(exported["particle_mean"]).all()
assert len(fig.axes)==3 and metrics["resampling_events"]>0
plt.close(fig)
print("Lab B filtering, ancestry, app wiring and CSV checks passed.")


The next cell launches the complete app in Colab. Keep the runtime active while using the temporary share link. Your permanent entry point is this notebook, not the temporary URL. Local users may run app.launch() when ready to open a local server.


In [ ]:
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    app.launch(share=True,inline=True,prevent_thread_lock=True)
else:
    print("Sampling Explorer is assembled. For local interactive use: app.launch()")
